In [2]:
# Celda 1: Instalación y prueba de conexión con Gemini
!uv pip install google-genai python-dotenv

from dotenv import load_dotenv
from google import genai

# Cargamos el archivo .env donde guardaste tu GEMINI_API_KEY
load_dotenv()

# Inicializamos el cliente oficial de Google
client = genai.Client()

# Enviamos nuestra primera consulta al modelo recomendado (Gemini 2.5 Flash)
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Hola Gemini, estoy configurando mi entorno RAG. ¿Está todo listo?',
)

print("Respuesta de Gemini:")
print(response.text)


Checked 2 packages in 4ms
Respuesta de Gemini:
¡Hola! ¡Absolutamente! Estoy listo para ayudarte con tu entorno RAG.

Como soy una IA y no tengo acceso directo a tu sistema, no puedo saber si *todo* está listo en tu configuración específica. Sin embargo, estoy aquí para asistirte en cada etapa del proceso.

Para que podamos empezar, dime:

1.  **¿En qué etapa de la configuración te encuentras?**
2.  **¿Qué componentes o herramientas has configurado hasta ahora (por ejemplo, base de datos vectorial, modelos de lenguaje, *chunking*, *embedding*, etc.)?**
3.  **¿Hay algo específico que te gustaría revisar o alguna pregunta que tengas?**
4.  **¿Estás enfrentando algún problema o error en particular?**

¡Dime dónde te has quedado o qué te gustaría revisar y te ayudaré con gusto!


In [3]:
# 1. Definimos la función 'llm' adaptada para Gemini
def llm(prompt):
    # Usamos el cliente nativo que inicializaste en el paso anterior
    response = client.models.generate_content(
        model='gemini-2.5-flash', # Reemplaza a gpt-5.4-mini
        contents=prompt           # Reemplaza a 'input=' de OpenAI
    )
    # Gemini devuelve el texto directamente en '.text'
    return response.text

# 2. Primera prueba: Comprobar la caja negra
print("Prueba de saludo:")
print(llm("Hey, what's up?"))
print("-" * 50)

# 3. Segunda prueba: Pregunta específica del curso
question = "I just discovered the course. Can I join now?"
answer = llm(question)

print("Pregunta del curso:")
print(answer)


Prueba de saludo:
Hey there!

Not much on my end, just here and ready to assist. How about you? What's up, or how can I help you today?
--------------------------------------------------
Pregunta del curso:
That's great news! We're happy you discovered the course.

Whether you can join now depends on a few factors:

1.  **Is it a self-paced, always-open course?** Many online courses allow you to enroll anytime and start immediately.
2.  **Is it a scheduled course with fixed start dates?** If so, you might have missed the current intake and may need to wait for the next cohort, or there might be a late enrollment period.
3.  **Are there limited spots?** Some courses, especially those with live interaction or specific resources, have limited capacity.
4.  **Is there an application process or prerequisites?**

**To give you the most accurate answer, I recommend you:**

*   **Check the official course page or website.** Look for sections like "Enrollment," "How to Join," "Admissions," "FAQ

In [4]:
# 1. Definimos el contexto extraído de las preguntas frecuentes (FAQ)
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

# Reutilizamos la variable 'question' que definiste en la celda anterior
question = "I just discovered the course. Can I join now?"

# 2. Construimos el prompt final combinando las instrucciones, la pregunta y el contexto
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

# 3. Enviamos el súper prompt a nuestra función de Gemini
answer = llm(prompt)

print("Respuesta de Gemini con RAG Manual:")
print(answer)


Respuesta de Gemini con RAG Manual:
Yes, you can still join. However, if you wish to receive a certificate, you will need to submit your project while submissions are still being accepted. You can also just start learning and submitting homework (while the form is open) without formal registration.


In [5]:
import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [6]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [7]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [8]:
from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents)

In [9]:
question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5},
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [17]:
results = index.search(
    question,
    num_results=5,
    boost_dict={"question": 2.0, "section": 0.5}
)


In [18]:
results = index.search(
    question,
    num_results=5,
    filter_dict={"course": "mlops-zoomcamp"}
)

In [19]:
[doc["question"] for doc in results]

['Course - Can I still join the course after the start date?',
 'Homework: Just found this course, can I still submit homeworks?',
 'I forgot if I registered, can I still join the zoomcamp?',
 'Certificate - Can I follow the course in a self-paced mode and get a certificate?',
 'Course: How do I start?']

In [20]:
def search(question, course="llm-zoomcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [21]:
search_results = search(question)

In [22]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [23]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [24]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [25]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [26]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [33]:
# Así se envía el prompt usando el cliente nativo de Google
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt
)



In [40]:
#The first one is the message:
response.candidates[0]

Candidate(
  content=Content(
    parts=[
      Part(
        text="""Yes, you can join now.

You can start learning whenever you want, as the videos and GitHub materials are available.

However, if you want to receive a certificate, you need to:
*   Submit your project while submissions are still being accepted.
*   Finish the course with the "live" cohort, as certificates are not awarded for self-paced mode. This is because you need to peer-review 3 capstone projects, which is only possible when the course is actively running and the peer-review list is compiled.

The next offering of the course will be in Summer 2027."""
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)

In [43]:
#The message has a content list, and the text is in the first item:
response.text
#response.candidates[0].content.parts[0].text

'Yes, you can join now.\n\nYou can start learning whenever you want, as the videos and GitHub materials are available.\n\nHowever, if you want to receive a certificate, you need to:\n*   Submit your project while submissions are still being accepted.\n*   Finish the course with the "live" cohort, as certificates are not awarded for self-paced mode. This is because you need to peer-review 3 capstone projects, which is only possible when the course is actively running and the peer-review list is compiled.\n\nThe next offering of the course will be in Summer 2027.'

In [44]:
#The usage counts tell you how many tokens the request consumed:
print(response.usage_metadata)


cache_tokens_details=None cached_content_token_count=None candidates_token_count=128 candidates_tokens_details=None prompt_token_count=514 prompt_tokens_details=[ModalityTokenCount(
  modality=<MediaModality.TEXT: 'TEXT'>,
  token_count=514
)] thoughts_token_count=668 tool_use_prompt_token_count=None tool_use_prompt_tokens_details=None total_token_count=1310 traffic_type=None


In [46]:
#Los precios de Gemini 2.5 Flash en la plataforma de Google AI Studio son considerablemente más económicos que los de GPT-5.4-mini
#costando únicamente $0.30 por millón de tokens de entrada y $2.50 por millón de tokens de salida.

# Definimos los precios oficiales de Gemini 2.5 Flash por millón de tokens
input_price = 0.30 / 1_000_000
output_price = 2.50 / 1_000_000

# Extraemos los tokens directamente del objeto de metadatos de Google
cost = (
    response.usage_metadata.prompt_token_count * input_price +
    response.usage_metadata.candidates_token_count * output_price
)

print(f"Costo de esta consulta: ${cost:.6f}")


Costo de esta consulta: $0.000474


In [49]:
#Message history
#Previously we sent only one string as input and got back a response. 
#In practice, you typically send a message history - a list of messages where each message has a role.
#Think of a ChatGPT conversation. It starts with a hidden system prompt that tells the LLM how to behave, one you never see. After that, your messages and the LLM's replies alternate. The LLM has no memory of its own, so it needs the full history passed in to continue the conversation.

# 1. Definimos las instrucciones fijas del sistema (System Prompt)
INSTRUCTIONS = """
You are a helpful assistant for the LLM Zoomcamp course. 
Answer the user's questions politely using only the provided context.
"""

# 2. Tu prompt con la pregunta y el contexto (el que armamos en la sección anterior)
# Asegúrate de haber corrido las celdas de 'prompt' y 'context' antes.

# 3. En Gemini nativo, las instrucciones del desarrollador VAN SEPARADAS en la configuración.
# Esto es equivalente a separar los roles 'developer' y 'user'.
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents=prompt,  # Este es el rol 'user'
    config={
        'system_instruction': INSTRUCTIONS  # Este es el rol 'developer' / 'system'
    }
)

# 4. Mostramos el resultado final obtenido
print("Respuesta con Instrucciones de Desarrollador:")
print(response.text)


Respuesta con Instrucciones de Desarrollador:
Yes, you can join now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [51]:
#The LLM function
# Definimos la función actualizada adaptada para Gemini
def llm(instructions, user_prompt, model="gemini-2.5-flash"):
    # Enviamos los parámetros a la API nativa de Google
    response = client.models.generate_content(
        model=model,
        contents=user_prompt, # El prompt del usuario va aquí
        config={
            'system_instruction': instructions # Las instrucciones del desarrollador van aquí
        }
    )
    
    # Retornamos el texto limpio de la respuesta
    return response.text


In [52]:
#Full RAG
# Definimos la función RAG completa adaptada para Gemini
def rag(query, model="gemini-2.5-flash"):
    # 1. Recuperación: Busca en tu base de datos (MinSearch/Elasticsearch)
    search_results = search(query)
    
    # 2. Aumento: Construye el súper prompt inyectando el contexto encontrado
    prompt = build_prompt(query, search_results)
    
    # 3. Generación: Envía las instrucciones y el prompt a Gemini
    answer = llm(INSTRUCTIONS, prompt, model=model)
    
    # Devuelve la respuesta final refinada
    return answer


In [53]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

Yes, you can join now. You can start whenever you want as the videos and GitHub materials are available.

However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [55]:
answer = rag("How do I get a certificate?")
print(answer)

To get a certificate, you must finish the course with a "live" cohort. You need to pass the Capstone project, and homework is not mandatory for this, though it is recommended.

Additionally, you need to peer-review 3 capstone projects after submitting your own. Peer-reviewing can only be done at the time the course is running, after the form is closed and the peer-review list is compiled.
